# 03 — Short-Term Demand Forecasting (RQ2)
*Can short-term energy demand be predicted?*

Day-ahead (24 h) forecasting: seasonal-naive and moving-average baselines vs
Ridge, Random Forest and LightGBM, evaluated with MAE / RMSE / MAPE / sMAPE
on a chronological hold-out window.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import plotly.express as px

from src.data.ingestion import load_dataset
from src.data.cleaning import clean_dataset

# Default: synthetic data (offline). Switch to source="bdg2" for real BDG2 data.
ds = load_dataset(source="synthetic", n_buildings=8)
print(f"{ds.meters.shape[0]:,} readings | {ds.metadata.shape[0]} buildings")
ds.metadata

In [ ]:
from src.features.engineering import build_feature_matrix
from src.models.forecasting import run_forecasting, best_model_per_building

clean, _ = clean_dataset(ds.meters)
features, feature_cols = build_feature_matrix(clean, ds.weather)
print(len(feature_cols), "features:", feature_cols)

In [ ]:
result = run_forecasting(features, feature_cols)
result.metrics.style.background_gradient(subset=["skill_vs_baseline_%"], cmap="RdYlGn")

## Best model per building

In [ ]:
best_model_per_building(result.metrics)

## Forecast vs actual — normal and difficult periods

In [ ]:
import plotly.graph_objects as go
b = result.predictions["building_id"].iloc[0]
p = result.predictions[result.predictions["building_id"] == b]
fig = go.Figure()
actual = p[p["model"] == "RandomForest"]
fig.add_scatter(x=actual["timestamp"], y=actual["actual"], name="Actual",
                line=dict(color="#444", width=2))
for m in ["SeasonalNaive24", "RandomForest"]:
    pm = p[p["model"] == m]
    fig.add_scatter(x=pm["timestamp"], y=pm["prediction"], name=m, line=dict(width=1.3))
fig.update_layout(title=f"Day-ahead forecast — {b}", yaxis_title="kW")

## Feature importance

In [ ]:
imp = result.feature_importance
imp = imp[(imp["building_id"] == b) & (imp["model"] == "RandomForest")].nlargest(12, "importance")
px.bar(imp, x="importance", y="feature", orientation="h").update_yaxes(autorange="reversed")

## Conclusions
- ML models clearly beat the seasonal-naive baseline (skill reported per building —
  honest comparison as required by the case study).
- Lag and rolling features dominate importance; temperature adds seasonal signal.
- Leakage guard: all lags ≥ forecast horizon (24 h).